In [1]:
import sys
import os
import pandas as pd

src_path = os.path.abspath(os.path.join(os.path.dirname("__file__"), '..', 'src'))
if src_path not in sys.path:
    sys.path.append(src_path)

from dense_index_bge import DenseIndex
from sparse_index import SparseIndex


libgomp: Invalid value for environment variable OMP_NUM_THREADS


In [2]:
court_consideration_df = pd.read_csv("../data/court_considerations.csv")
court_consideration_d = dict(zip(court_consideration_df['citation'].tolist(), court_consideration_df['text'].tolist()))

law_df = pd.read_csv("../data/laws_de.csv")
law_d = dict(zip(law_df['citation'].tolist(), law_df['text'].tolist()))

court_doc = [{'citation':citation, 'text':text} for citation,text in zip(court_consideration_df['citation'].tolist(), court_consideration_df['text'].tolist())]
law_doc = [{'citation':citation, 'text':text} for citation,text in zip(law_df['citation'].tolist(), law_df['text'].tolist())]

test_df = pd.read_csv("../data/test_rewrite_001.csv")
_d = {}
for _, row in test_df.iterrows():
    if row['query_id'] not in _d:
        _d[row['query_id']] = [row['query']]
    else:
        _d[row['query_id']].append(row['query'])
test_dict = {k: v for k, v in sorted(_d.items())}

valid_df = pd.read_csv("../data/valid_rewrite_001.csv")

print("data loaded.")

data loaded.


In [3]:
from FlagEmbedding import FlagReranker, BGEM3FlagModel

model = BGEM3FlagModel('/root/.cache/modelscope/hub/models/BAAI/bge-m3', use_fp16=True, show_progress_bar=False)

dense_index = DenseIndex(model, "/root/autodl-fs/bge-processed/_dense_sparse_court/", court_doc)

sparse_index = SparseIndex(model, "/root/autodl-fs/bge-processed/_dense_sparse_court/", court_doc)


libgomp: Invalid value for environment variable OMP_NUM_THREADS

libgomp: Invalid value for environment variable OMP_NUM_THREADS


DenseIndex.embeddings:  (2107648, 1024)


In [22]:
reranker = FlagReranker('/root/.cache/modelscope/hub/models/BAAI/bge-reranker-v2-m3', use_fp16=True, normalize=True)
# reranker = FlagReranker('../ft_data/merged_reranker', use_fp16=True, normalize=True)

In [5]:
valid_df = pd.read_csv("../data/valid_rewrite_001.csv")
valid_id_2_gold_citations_d = {}
valid_id_2_query_d = {}
for query_id, gold_citations, query in zip(valid_df['query_id'], valid_df['gold_citations'], valid_df['query2']):
    valid_id_2_gold_citations_d[query_id] = gold_citations.split(';')
    valid_id_2_query_d[query_id] = query

In [19]:
%load_ext autoreload

%autoreload 2

import metric_utils
import citation_utils
import hits_utils
import reranker_utils
import rrf
from collections import defaultdict
from tqdm.notebook import tqdm

valid_multi_question_df = pd.read_csv("../data/valid_rewrite_split_question_001.csv")

recall_citations_l = []
gold_citations_l = []

query_id_2_query_list = defaultdict(list)

for query_id, query in zip(valid_multi_question_df['query_id'], valid_multi_question_df['query']):
    query_id_2_query_list[query_id].append(query)

for query_id, query in valid_id_2_query_d.items():
    query_id_2_query_list[query_id].append(query) # 完整的问题在最后一个

recall_hits_l = []
for query_id, query_list in query_id_2_query_list.items():
    all_hits = []
    for query in query_list:
        hits1 = dense_index.search_with_score(query, top_k=200)
        hits2 = sparse_index.search_with_score(query, top_k=200)
        hits_merge = hits_utils.merge_hits_with_score_l_by_max(hits1, hits2)
        all_hits = hits_utils.merge_hits_with_score_l_by_max(all_hits, hits_merge)

    print(f"[{query_id}] hits_merge.len:", len(all_hits))
    gold_citations_l.append(valid_id_2_gold_citations_d[query_id])

    # neighbour_hit = []
    # for hit, score in all_hits:
    #     hits1 = dense_index.search_with_score(hit['text'], top_k=5)
    #     neighbour_hit.extend(hits1)

    # all_hits.extend(neighbour_hit)

    recall_hits_l.append([hit for hit, score in all_hits])
    
    second_layer = citation_utils.compute_citation_score_with_sentence_pos(all_hits, decay="log")

    citations = []
    for citation, score in second_layer:
        if citation in court_consideration_d:
            citations.append(citation)
        if citation in law_d:
            citations.append(citation)

    recall_citations_l.append(list(set(citations)))
    
r = metric_utils.cal_recall(recall_citations_l, gold_citations_l)
print(r)

# 准备好document
# recall_hits_l = []
# for recall_citations in recall_citations_l:
#     recall_hits = []
#     for citation in set(recall_citations):
#         if citation in court_consideration_d:
#             recall_hits.append({'citation':citation, 'text':court_consideration_d[citation]})
#         elif citation in law_d:
#             recall_hits.append({'citation':citation, 'text':law_d[citation]})
#     recall_hits_l.append(recall_hits)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
[val_001] hits_merge.len: 579
[val_002] hits_merge.len: 972
[val_003] hits_merge.len: 854
[val_004] hits_merge.len: 665
[val_005] hits_merge.len: 878
[val_006] hits_merge.len: 668
[val_007] hits_merge.len: 1035
[val_008] hits_merge.len: 1370
[val_009] hits_merge.len: 851
[val_010] hits_merge.len: 886
0.4285548447253665


In [20]:
import numpy as np

def logsumexp(scores):
    scores = np.array(scores)
    m = np.max(scores)
    return m + np.log(np.sum(np.exp(scores - m)))

In [23]:
%load_ext autoreload

%autoreload 2

import metric_utils
import citation_utils
import hits_utils
import reranker_utils
import rrf
from collections import defaultdict
from tqdm.notebook import tqdm

query_id_l = []
result_citation_l = []

for idx, (query_id, query_list) in tqdm(enumerate(query_id_2_query_list.items()), total=len(query_id_2_query_list)):
    # 计算evidence分数
    evidence_l = []
    for hit in recall_hits_l[idx]:
        cc = citation_utils.parse_cc_output_citations_and_sentences(hit['text'])
        for citation, first_appears_idx in cc['citations']:
            evidence = citation_utils.build_evidence(cc['sentences'], first_appears_idx, window_size=5)
            evidence_l.append({'citation':citation, 'text':evidence})

    print("===>",query_id, ", evidence_l.len:", len(evidence_l))

    raw_query = query_list[-1]
    normalized_query_list = query_list[:-1] # 最后一个包含多个问题
    question_l = []
    for q in normalized_query_list:
        _, question = q.rsplit('\n\n', 1)
        question_l.append(question)
    
    evidence_with_score_l_raw = reranker_utils.rerank_by_batch_chunked2(reranker, raw_query, evidence_l)
    # evidence_with_score_l_question = reranker_utils.rerank_by_batch_chunked2(reranker, '\n\n'.join(question_l), evidence_l)

    # evidence_with_score_l = [(raw_score[0], raw_score[1]+0.5*question_score[1]) for raw_score, question_score in zip(evidence_with_score_l_raw, evidence_with_score_l_question)]
    evidence_with_score_l = evidence_with_score_l_raw
    # agg by max
    # citation_score_d = defaultdict(float)
    # for evidence, score in evidence_with_score_l:
    #     if evidence['citation'] not in citation_score_d:
    #         citation_score_d[evidence['citation']] = score
    #     elif citation_score_d[evidence['citation']] < score:
    #         citation_score_d[evidence['citation']] = score

    # agg by sum
    # citation_score_d = defaultdict(float)
    # for evidence, score in evidence_with_score_l:
    #     citation_score_d[evidence['citation']] += score

    # agg by top-k score sum
    citation_evidence_score_l_d = defaultdict(list)
    for evidence, score in evidence_with_score_l:
        citation_evidence_score_l_d[evidence['citation']].append((evidence, score))

    citation_score_d = {}
    for citation, l in citation_evidence_score_l_d.items():
        l2 = sorted(l, key=lambda x: x[1], reverse=True)[:5]
        score = sum([score for _, score in l2])
        citation_score_d[citation] = score
    
    # 对综合分数排序
    sorted_citation_score_l = sorted([(c,score) for c,score in citation_score_d.items()], key=lambda x: x[1], reverse=True)
    
    # 将citation合并
    query_result = [citation for citation, _ in sorted_citation_score_l]
    
    query_result2 = [citation for citation in query_result if citation in court_consideration_d or citation in law_d]
    
    result_citation_l.append(query_result2)

    print("result_citation_l[-1].len:", len(result_citation_l[-1]))

    
max_limit = None
max_r = None
max_p = None
max_f1 = 0
for limit in [5,10,15,20,25,30,35,40,45]:
    result_citation_l2 = [r[:limit] for r in result_citation_l]
    result = metric_utils.macro_f1(result_citation_l2, gold_citations_l[:len(result_citation_l2)])
    if max_f1 < result['macro_f1']:
        max_r = result['macro_recall']
        max_p = result['macro_precision']
        max_limit = limit
        max_f1 = result['macro_f1']
print(f"[{max_limit}] r:", max_r, ", p:", max_p, "f1:",max_f1)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


  0%|          | 0/10 [00:00<?, ?it/s]

===> val_001 , evidence_l.len: 1777
result_citation_l[-1].len: 167
===> val_002 , evidence_l.len: 2104
result_citation_l[-1].len: 197
===> val_003 , evidence_l.len: 2383
result_citation_l[-1].len: 281
===> val_004 , evidence_l.len: 1411
result_citation_l[-1].len: 247
===> val_005 , evidence_l.len: 3123
result_citation_l[-1].len: 444
===> val_006 , evidence_l.len: 2215
result_citation_l[-1].len: 367
===> val_007 , evidence_l.len: 4107
result_citation_l[-1].len: 788
===> val_008 , evidence_l.len: 5456
result_citation_l[-1].len: 762
===> val_009 , evidence_l.len: 1870
result_citation_l[-1].len: 297
===> val_010 , evidence_l.len: 2395
result_citation_l[-1].len: 415
[10] r: 0.17178800666710464 , p: 0.32 f1: 0.21259124052779854
